# Fitting a steady point-source with the public 14-year IceCube track data

This tutorial shows how to fit a neutrino point-like steady source (time-integrated) using the IceCube public 14-year point-source data with SkyLLH.
At the end of the notebook we also show how the same fit works using the older 10-year dataset.

We assume that you have installed the SkyLLH package in your environment. If that's not the case you'll need to add the path to the SkyLLH folder to your PYTHONPATH.

In [1]:
import numpy as np

## Initializing a configuration dictionary

First we have to create a config instance.

In [2]:
from skyllh.core.config import Config

cfg = Config()

The Config object stores a dictionary with global settings, including computing, logging, and analysis settings. It is a required arugment to several methods.

In [3]:
cfg

{'multiproc': {'ncpu': None},
 'logging': {'log_level': 'INFO',
  'log_format': '%(asctime)s %(processName)s %(name)s %(levelname)s: %(message)s',
  'enable_tracing': False},
 'project': {'working_directory': '.'},
 'repository': {'base_path': '/Users/chiarabellenghi/.cache/skyllh',
  'download_from_origin': True},
 'units': {'internal': {'angle': Unit("rad"),
   'energy': Unit("GeV"),
   'length': Unit("cm"),
   'time': Unit("s")},
  'defaults': {'fluxes': {'angle': Unit("rad"),
    'energy': Unit("GeV"),
    'length': Unit("cm"),
    'time': Unit("s")}}},
 'datafields': {'run': 4,
  'ra': 4,
  'dec': 4,
  'ang_err': 4,
  'time': 4,
  'log_energy': 4,
  'true_ra': 8,
  'true_dec': 8,
  'true_energy': 8,
  'mcweight': 8},
 'caching': {'pdf': {'MultiDimGridPDF': False}}}

## Setting up logging

The user sets up the main logger. The logger name, logging format and level, output file, and other options can be set.

Default logging format and level are defined in the Config instance, but they can be overwritten by the arguments of `logging.setup_loggging`.

In [ ]:
from skyllh.core.logging import setup_logging

logger = setup_logging(cfg=cfg, name='fitting_a_source', log_level='INFO')

## Getting the datasets

The datasets are automatically downloaded from [dataverse.harvard.edu](https://dataverse.harvard.edu/dataverse/icecube) to a local cache directory (`~/.skyllh/cache`) on first run. To use a custom dataset location, set `cfg['repository']['base_path']` to the desired path.

To create the correct dataset object, we need to import the 14-yr dataset definition:

In [5]:
import skyllh

The collection of datasets is created with `create_dataset_collection`. With the default configuration, the data is downloaded automatically from `dataverse.harvard.edu`.

In [9]:
help(skyllh.create_datasets)

Help on function create_datasets in module skyllh.datasets.datasets:

create_datasets(sample_name, cfg, names=None, base_path=None, sub_path_fmt=None)
    Creates a list of Dataset instances for a named data sample.

    Parameters
    ----------
    sample_name : str
        The name of the data sample. Available samples are the keys of
        ``skyllh.datasets.data_samples``.
    cfg : instance of Config
        The instance of Config holding the local configuration.
    names : sequence of str | None
        The dataset names to return. If None, the module's ``DATASET_NAMES``
        default is used (combined IC86 seasons where applicable).
    base_path : str | None
        The base path of the data files. If None, uses
        ``cfg['repository']['base_path']``.
    sub_path_fmt : str | None
        The sub path format override. If None, uses the module default.

    Returns
    -------
    datasets : list of Dataset



If no `names` are passed to `skyllh.create_datasets`, it loads all data sets that are part of the IceTracks-DR2 dataset collection.

A tutorial on how to access specific information about which datasets are included in, e.g., IceTracks-DR2 is provided in ..todo::`dataset_collections` tutorial.

In [ ]:
datasets = skyllh.create_datasets('IceTracks-DR2', cfg=cfg)

By default, this imports all available data sets in IceTracks-DR2 (all 14 years of data).

## Creating the analysis object

The analysis for doing a point-source search as in, e.g., https://doi.org/10.1103/PhysRevLett.124.051103 is pre-defined. This analysis instance can be created via the `create_analysis` method of the `time_integrated_ps` module of the public data interface.

The function requires the config, the datasets, and a source object. It also takes a number of additional optional arguments for which we refer the reader to the docstring.

In [11]:
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis

help(create_analysis)

Help on function create_analysis in module skyllh.analyses.i3.publicdata_ps.time_integrated_ps:

create_analysis(
    cfg,
    datasets,
    source,
    refplflux_Phi0=1,
    refplflux_E0=1000.0,
    refplflux_gamma=2.0,
    refplflux_Ec=inf,
    energy_range=None,
    ns_seed=10.0,
    ns_min=0.0,
    ns_max=1000.0,
    gamma_seed=3.0,
    gamma_min=1.0,
    gamma_max=4.0,
    kde_smoothing=False,
    minimizer_impl='LBFGS',
    minimizer_max_rep=100,
    compress_data=False,
    keep_data_fields=None,
    evt_sel_delta_angle_deg=10,
    construct_sig_generator=True,
    tl=None,
    ppbar=None,
    logger_name=None
)
    Creates the Analysis instance for this particular analysis.

    Parameters
    ----------
    cfg : instance of Config
        The instance of Config holding the local configuration.
    datasets : list of Dataset instances
        The list of Dataset instances, which should be used in the
        analysis.
    source : PointLikeSource instance
        The PointLike

Defining the Source: Here we use the location of NGC 1068 as an example source.

Instantiating a `PointLikeSource` object required the source location (R.A. and Dec.) in radians. Optionally, the source can have a `name` and a `weight`. The latter is in principle useful when performing a stacking analysis which is not supported for public data yet.

In [15]:
from skyllh.core.source_model import PointLikeSource

help(PointLikeSource)

Help on class PointLikeSource in module skyllh.core.source_model:

class PointLikeSource(SourceModel, IsPointlike)
 |  PointLikeSource(ra, dec, name=None, weight=None, **kwargs)
 |
 |  The PointLikeSource class is a source model for a point-like source
 |  object in the sky at a given location (right-ascention and declination).
 |
 |  Method resolution order:
 |      PointLikeSource
 |      SourceModel
 |      skyllh.core.model.Model
 |      IsPointlike
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, ra, dec, name=None, weight=None, **kwargs)
 |      Creates a new PointLikeSource instance for defining a point-like
 |      source.
 |
 |      Parameters
 |      ----------
 |      ra : float
 |          The right-ascention coordinate of the source in radians.
 |      dec : float
 |          The declination coordinate of the source in radians.
 |      name : str | None
 |          The name of the source.
 |      weight : float | None
 |          The relative weig

In [16]:
src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(ra=np.radians(src_ra), dec=np.radians(src_dec))

If not otherwise specified through the optional arguments, the `time_integrated_ps.create_analysis` method source assumes a signal hypothesis such that the defined source emits a powerlaw flux with spectral index $\gamma=2.0$:

$ \phi(E_{\nu}) = \mathrm{refplflux\_Phi0} \cdot \left(\frac{E_{\nu}}{\mathrm{refplflux\_E0}}\right)^{-\mathrm{refplflux\_gamma}}$

In [17]:
ana = create_analysis(cfg=cfg, datasets=datasets, source=source)

2026-05-13 14:49:35,082 MainProcess skyllh.analyses.i3.publicdata_ps.time_integrated_ps INFO: SourceHypoGroupManager
    Source Hypothesis Groups:
        0: SourceHypoGroup:
            sources (1):
                0: PointLikeSource: "4590251664": { ra=40.670 deg, dec=-0.010 deg }
            fluxmodel:
                1.000e+00 * (E / (1000 GeV))^-2 * 1 (GeV cm^2 s)^-1
            detector signal yield builders (1):
                PDSingleParamFluxPointLikeSourceI3DetSigYieldBuilder
            signal generation method:
                NoneType
2026-05-13 14:49:35,083 MainProcess skyllh.analyses.i3.publicdata_ps.time_integrated_ps INFO: ParameterModelMapper: 2 global parameters, 2 models (1 source)
    Parameters:        
        ns [floating (0 <= 10 <= 1000)]
            in models:
            - IceCube: ns
                    
        gamma [floating (1 <= 3 <= 4)]
            in models:
            - 4590251664: gamma
            
  0%|          | 0/4 [00:00<?, ?it/s]2026-05-13

DatasetTransferError: HTTP Error 401: Unauthorized

## Understanding how the analysis works

One usually wants to get best-fit values (maximum likelihood estimators) for the source parameters and a test-statistic, using methods like `ana.do_trial` or `ana.unblind` (see below). However, here we describe what happens under the hood when the analysis is run either on simulated or real experimental data.

### Initializing a trial

After the `Analysis` instance was created, the point-source analysis can be run. To do so the analysis is initialized with some _trial data_.
For instance we could initialize the analysis with the experimental data but the same applies for simulated data. 

The `Analysis` object provides the method `initialize_trial` to initialize a trial with data. It takes a list of `DataFieldRecordArray` instances holding the events. If we want to initialize a trial with the experimental data, we can get that list from the `Analysis` instance itself:

In [19]:
events_list = [data.exp for data in ana.data_list]
ana.initialize_trial(events_list)

NameError: name 'ana' is not defined

### Maximizing the log-likelihood ratio function

After initializing a trial, we can maximize the LLH ratio function using the `ana.llhratio.maximize` method. This method requires a ``RandomStateService`` instance in case the minimizer does not succeed and a new set of initial values for the fit parameters need to get generated. The method returns a 3-element tuple. The first element is the value of the LLH ratio function at its maximum. The second element is the set of fit parameters that maximize the LLH ratio. The third element is the status dictionary of the minimizer.

In [20]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

In [11]:
(log_lambda_max, fitparam_values, status) = ana.llhratio.maximize(rss)

In [12]:
print(f'log_lambda_max = {log_lambda_max}')
print(f'fitparam_values = {fitparam_values}')
print(f'status = {status}')

log_lambda_max = 14.576758408087755
fitparam_values = [80.09381868  3.20884803]
status = {'grad': array([-3.62369895e-06, -2.31360821e-04]), 'task': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH', 'funcalls': 13, 'nit': 10, 'warnflag': 0, 'skyllh_minimizer_n_reps': 0, 'n_llhratio_func_calls': 13}


### Calculating the test-statistic

Using the maximum of the LLH ratio function and the fit parameter values at the maximum we can calculate the test-statistic using the `ana.calculate_test_statistic` method:

In [13]:
TS = ana.calculate_test_statistic(log_lambda_max, fitparam_values)
print(f'TS = {TS:.3f}')

TS = 29.154


## Unblinding the data

Above, we perform the analysis following pedagogical steps: we initialize the analysis with a trial of the experimental data, maximize the log-likelihood ratio function for all given experimental data events, and calculate the test-statistic value.

However, the same calculations can be performed all togethe using the ``ana.unblind`` method of the analysis instance, which returns the test statistic and the best fit parameters directly.

Note that the ``ana.unblind`` method runs the analysis on the experimental data.

The methods to generate and analyze pseudo-data are illustrated in the `generating_pseudo_experiments.ipynb` notebook.

In [17]:
help(ana.unblind)

Help on method unblind in module skyllh.core.analysis:

unblind(minimizer_rss, tl=None) method of skyllh.core.analysis.SingleSourceMultiDatasetLLHRatioAnalysis instance
    Evaluates the unscrambled data, i.e. unblinds the data.

    Parameters
    ----------
    minimizer_rss : instance of RandomStateService
        The instance of RandomStateService that should be used by the
        minimizer to generate new random initial fit parameter values.
    tl : instance of TimeLord | None
        The optional instance of TimeLord that should be used to time the
        maximization of the LLH ratio function.

    Returns
    -------
    TS : float
        The test-statistic value.
    global_params_dict : dict
        The dictionary holding the global parameter names and their
        best fit values. It includes fixed and floating parameters.
    status : dict
        The status dictionary with information about the performed
        minimization process of the negative of the log-likeliho

In [ ]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

(ts, fitparam_values, status) = ana.unblind(rss)

In [ ]:
print(f'TS = {ts:.3f}')
print(f'ns = {fitparam_values["ns"]:.2f}')
print(f'gamma = {fitparam_values["gamma"]:.2f}')
status

TS = 29.154
ns = 80.09
gamma = 3.21


{'grad': array([-3.62369979e-06, -2.31360814e-04]),
 'task': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH',
 'funcalls': 13,
 'nit': 10,
 'warnflag': 0,
 'skyllh_minimizer_n_reps': 0,
 'n_llhratio_func_calls': 13}

## Converting the fit parameters into a flux normalization 

..todo: Tomas can you please a function that takes a specific flux param to get the conversion factor?

..todo: Further utility functions are available, 
1. ``ana.mu2flux(mu)`` returns a flux normalization at the chosen spectral index, chosen during the initialisation of the analysis instance. 
2. ``ana.flux2mu(flux_norm)`` returns the mean number of signal events for the given flux normalization value at the chosen spectral index.

By default the analysis is created with a flux normalization of 1 $\text{GeV}^{-1}\text{s}^{-1}\text{cm}^{-2}\text{sr}^{-1}$ at 1 TeV (see `refplflux_Phi0` and `refplflux_E0` arguments of the `create_analysis` method). The analysis instance has the method `calculate_fluxmodel_scaling_factor` that calculates the scaling factor the reference flux normalization has to be multiplied with to represent a given analysis result, i.e. $n_{\text{s}}$ and $\gamma$ value. This function takes the detected mean $n_{\text{s}}$ value as first argument and the list of source parameter values as second argument:

In [ ]:
scaling_factor = ana.calculate_fluxmodel_scaling_factor(
    fitparam_values['ns'], [fitparam_values['ns'], fitparam_values['gamma']]
)
print(f'Flux scaling factor = {scaling_factor:.3e}')

Flux scaling factor = 2.869e-14


Hence, our result corresponds to a power-law flux of:

In [ ]:
print(f'{scaling_factor:.3e} (E/1000 GeV)^{{ -{fitparam_values["gamma"]:.2f} }} 1/(GeV s cm^2 sr)')

2.869e-14 (E/1000 GeV)^{-3.21} 1/(GeV s cm^2 sr)


Illustrating the difference of fits between the 10-year and the 14-year dataset. To compute the test statistic and the best fit parameters for the 10-year dataset, the same workflow as used for the 14-year dataset.  

In [22]:
from skyllh.core.config import Config

cfg_10 = Config()

In [23]:
from skyllh.datasets.i3.PublicData_10y_ps import create_dataset_collection

In [ ]:
dsc = create_dataset_collection(cfg=cfg_10)

datasets = dsc['IC40', 'IC59', 'IC79', 'IC86_I', 'IC86_II-VII']

In [25]:
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis
from skyllh.core.source_model import PointLikeSource

src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(ra=np.radians(src_ra), dec=np.radians(src_dec))
ana = create_analysis(cfg=cfg_10, datasets=datasets, source=source)

events_list = [data.exp for data in ana.data_list]

100%|██████████| 170/170 [00:00<00:00, 509.21it/s]


In [ ]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

(ts, x, status) = ana.unblind(rss)
print(f'TS = {ts:.3f}')
print(f'ns = {x["ns"]:.2f}')
print(f'gamma = {x["gamma"]:.2f}')
## Calculating the corresponding flux normalization
scaling_factor = ana.calculate_fluxmodel_scaling_factor(x['ns'], [x['ns'], x['gamma']])
print(f'Flux scaling factor = {scaling_factor:.3e}')

print(f'{scaling_factor:.3e} (E/1000 GeV)^{{ -{x["gamma"]:.2f} }} 1/(GeV s cm^2 sr)')

TS = 19.379
ns = 54.75
gamma = 3.16
Flux scaling factor = 3.017e-14
3.017e-14 (E/1000 GeV)^{-3.16} 1/(GeV s cm^2 sr)
